# **BÁO CÁO GIỮA KÌ MÔN KHAI THÁC DỮ LIỆU VÀ KHAI PHÁ TRI THỨC**
## **Đề tài: Khai thác luật kết hợp: cửa hàng bách hoá, siêu thị - Khởi tạo và Chuẩn hoá dữ liệu với bộ dữ liệu Instacart**

**Sinh viên:** [Điền họ tên]  
**MSSV:** [Điền MSSV]  
**Lớp / Nhóm:** [Điền lớp hoặc nhóm]  

---

### **Mục tiêu báo cáo**
Báo cáo này tập trung vào 4 nội dung chính:

- Khởi tạo dữ liệu từ nhiều bảng trong bộ dữ liệu Instacart
- Khảo sát và đánh giá chất lượng dữ liệu ban đầu
- Chuẩn hoá dữ liệu theo đúng ngữ nghĩa nghiệp vụ
- Tích hợp dữ liệu theo hướng tiết kiệm tài nguyên
- Biến đổi dữ liệu sang dạng giao dịch để tạo đầu vào trực tiếp cho giai đoạn cuối kì

### **Phạm vi thực hiện**
Trong phạm vi bài giữa kì, nội dung chỉ tập trung vào:

1. Khởi tạo dữ liệu  
2. Khảo sát và đánh giá chất lượng dữ liệu ban đầu  
3. Chuẩn hoá dữ liệu  
4. Tích hợp dữ liệu sau chuẩn hoá  

**Không mở rộng sang**:
- Khai thác luật kết hợp
- Phân tích giỏ hàng
- Xây dựng mô hình dự đoán

## **1. Giới thiệu bộ dữ liệu**

Bộ dữ liệu sử dụng trong báo cáo là **Instacart Market Basket Analysis**, gồm nhiều bảng dữ liệu quan hệ mô tả hành vi mua sắm của khách hàng trong siêu thị trực tuyến.

### **Các file dữ liệu chính**
- `orders.csv`: chứa thông tin đơn hàng
- `order_products__prior.csv`: chứa chi tiết sản phẩm trong các đơn hàng lịch sử
- `order_products__train.csv`: chứa chi tiết sản phẩm trong tập train
- `products.csv`: danh mục sản phẩm
- `aisles.csv`: danh mục quầy hàng
- `departments.csv`: danh mục ngành hàng


## **2. Khởi tạo môi trường**
Tiến hành import thư viện cần thiết và xây dựng một số hàm hỗ trợ để hiển thị bảng kết quả đẹp, rõ ràng hơn.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

In [2]:
def dinh_dang_bang(df, tieu_de=None):
    return (
        df.style
        .set_caption(tieu_de if tieu_de else "")
        .set_properties(**{
            "border": "1px solid #4f4f4f",
            "text-align": "center",
            "padding": "6px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#d9d9d9"),
                    ("color", "black"),
                    ("border", "1px solid #4f4f4f"),
                    ("text-align", "center"),
                    ("font-weight", "bold")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #4f4f4f")
                ]
            },
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "16px"),
                    ("font-weight", "bold"),
                    ("margin-bottom", "8px")
                ]
            }
        ])
    )

def tom_tat_bang(df, ten_bang):
    return {
        "Tên bảng": ten_bang,
        "Số dòng": df.shape[0],
        "Số cột": df.shape[1],
        "Giá trị thiếu": int(df.isnull().sum().sum()),
        "Dòng trùng": int(df.duplicated().sum())
    }

## **3. Đọc dữ liệu gốc**
Trong bước này, dữ liệu được đọc trực tiếp từ các file `.csv` để tiến hành khảo sát ban đầu.

In [3]:
import os
import pandas as pd

DATA_DIR = r"D:\HK2_NAM3\KTDL_KPTT\midterm\archive"

orders = pd.read_csv(os.path.join(DATA_DIR, "orders.csv"))
order_products_prior = pd.read_csv(os.path.join(DATA_DIR, "order_products__prior.csv"))
order_products_train = pd.read_csv(os.path.join(DATA_DIR, "order_products__train.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
aisles = pd.read_csv(os.path.join(DATA_DIR, "aisles.csv"))
departments = pd.read_csv(os.path.join(DATA_DIR, "departments.csv"))

print("Đã nạp thành công toàn bộ các bảng dữ liệu gốc.")

Đã nạp thành công toàn bộ các bảng dữ liệu gốc.


### **Nhận xét**
Việc đọc dữ liệu thành công cho thấy các file đầu vào đã được chuẩn bị đúng tên, đúng định dạng và sẵn sàng cho quá trình khảo sát chất lượng dữ liệu ban đầu.

## **4. Khảo sát quy mô các bảng dữ liệu**
Trước khi làm sạch dữ liệu, cần xác định số lượng dòng và số lượng cột của từng bảng để đánh giá quy mô và vai trò của từng tập dữ liệu.

In [4]:
tong_quan_kich_thuoc = pd.DataFrame([
    {"Tên bảng": "orders", "Số dòng": orders.shape[0], "Số cột": orders.shape[1]},
    {"Tên bảng": "order_products__prior", "Số dòng": order_products_prior.shape[0], "Số cột": order_products_prior.shape[1]},
    {"Tên bảng": "order_products__train", "Số dòng": order_products_train.shape[0], "Số cột": order_products_train.shape[1]},
    {"Tên bảng": "products", "Số dòng": products.shape[0], "Số cột": products.shape[1]},
    {"Tên bảng": "aisles", "Số dòng": aisles.shape[0], "Số cột": aisles.shape[1]},
    {"Tên bảng": "departments", "Số dòng": departments.shape[0], "Số cột": departments.shape[1]},
])

display(dinh_dang_bang(tong_quan_kich_thuoc, "Bảng 1. Tổng quan kích thước các bảng dữ liệu"))

,Tên bảng,Số dòng,Số cột
0,orders,3421083,7
1,order_products__prior,32434489,4
2,order_products__train,1384617,4
3,products,49688,4
4,aisles,134,2
5,departments,21,2


### **Nhận xét**
Kết quả cho thấy bộ dữ liệu Instacart có quy mô rất lớn, đặc biệt là bảng `order_products__prior` với hàng chục triệu bản ghi.  
Điều này cho thấy bộ dữ liệu đủ lớn để phục vụ các bài toán khai thác dữ liệu thực tế, đồng thời cũng yêu cầu quy trình xử lý dữ liệu phải hợp lý và tiết kiệm bộ nhớ.

## **5. Khảo sát cấu trúc cột dữ liệu**
Tiếp theo, tiến hành kiểm tra tên cột của từng bảng để xác định vai trò của từng thuộc tính và mối liên hệ giữa các bảng.

In [5]:
cau_truc_cot = pd.DataFrame({
    "Tên bảng": ["orders", "order_products__prior", "order_products__train", "products", "aisles", "departments"],
    "Danh sách cột": [
        ", ".join(orders.columns),
        ", ".join(order_products_prior.columns),
        ", ".join(order_products_train.columns),
        ", ".join(products.columns),
        ", ".join(aisles.columns),
        ", ".join(departments.columns)
    ]
})

display(dinh_dang_bang(cau_truc_cot, "Bảng 2. Cấu trúc cột của các bảng dữ liệu"))

,Tên bảng,Danh sách cột
0,orders,"order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order"
1,order_products__prior,"order_id, product_id, add_to_cart_order, reordered"
2,order_products__train,"order_id, product_id, add_to_cart_order, reordered"
3,products,"product_id, product_name, aisle_id, department_id"
4,aisles,"aisle_id, aisle"
5,departments,"department_id, department"


### **Nhận xét**
Kết quả cho thấy dữ liệu được tổ chức theo mô hình quan hệ nhiều bảng.  
Các khóa như `order_id`, `product_id`, `aisle_id`, `department_id` đóng vai trò liên kết giữa các bảng, tạo điều kiện thuận lợi cho quá trình tích hợp dữ liệu ở các bước sau.

## **6. Kiểm tra kiểu dữ liệu và xem mẫu dữ liệu**
Ở bước này, tiến hành kiểm tra kiểu dữ liệu của các cột và xem một vài dòng dữ liệu để hiểu rõ hơn về nội dung thực tế của từng bảng.

In [6]:
thong_tin_kieu_du_lieu = pd.DataFrame({
    "Bảng": ["orders", "order_products__prior", "products", "aisles", "departments"],
    "Kiểu dữ liệu": [
        ", ".join([f"{col}: {dtype}" for col, dtype in orders.dtypes.items()]),
        ", ".join([f"{col}: {dtype}" for col, dtype in order_products_prior.dtypes.items()]),
        ", ".join([f"{col}: {dtype}" for col, dtype in products.dtypes.items()]),
        ", ".join([f"{col}: {dtype}" for col, dtype in aisles.dtypes.items()]),
        ", ".join([f"{col}: {dtype}" for col, dtype in departments.dtypes.items()]),
    ]
})

display(dinh_dang_bang(thong_tin_kieu_du_lieu, "Bảng 3. Kiểu dữ liệu của các bảng chính"))

,Bảng,Kiểu dữ liệu
0,orders,"order_id: int64, user_id: int64, eval_set: str, order_number: int64, order_dow: int64, order_hour_of_day: int64, days_since_prior_order: float64"
1,order_products__prior,"order_id: int64, product_id: int64, add_to_cart_order: int64, reordered: int64"
2,products,"product_id: int64, product_name: str, aisle_id: int64, department_id: int64"
3,aisles,"aisle_id: int64, aisle: str"
4,departments,"department_id: int64, department: str"


In [7]:
display(Markdown("### Mẫu dữ liệu bảng `orders`"))
display(orders.head())

display(Markdown("### Mẫu dữ liệu bảng `order_products__prior`"))
display(order_products_prior.head())

display(Markdown("### Mẫu dữ liệu bảng `products`"))
display(products.head())

### Mẫu dữ liệu bảng `orders`

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


### Mẫu dữ liệu bảng `order_products__prior`

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


### Mẫu dữ liệu bảng `products`

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


### **Nhận xét**
Kiểu dữ liệu của các cột là hợp lý.  
Các cột mã định danh như `order_id`, `user_id`, `product_id` có kiểu số nguyên, các cột mô tả như tên sản phẩm hoặc nhóm ngành hàng ở dạng văn bản.  
Việc xem mẫu dữ liệu cũng giúp xác nhận rằng mỗi dòng trong `order_products__prior` tương ứng với một sản phẩm xuất hiện trong một đơn hàng.
Việc xem mẫu dữ liệu cũng giúp nhóm hiểu rõ hơn cấu trúc của từng bảng, đặc biệt là bảng order_products__prior, trong đó mỗi dòng tương ứng với một sản phẩm trong một đơn hàng.

## **7. Kiểm tra giá trị thiếu**
Kiểm tra dữ liệu thiếu giúp phát hiện các cột có vấn đề cần xử lý ở bước chuẩn hoá.

In [8]:
bang_missing = pd.DataFrame({
    "Bảng": ["orders", "order_products__prior", "order_products__train", "products", "aisles", "departments"],
    "Số giá trị thiếu": [
        int(orders.isnull().sum().sum()),
        int(order_products_prior.isnull().sum().sum()),
        int(order_products_train.isnull().sum().sum()),
        int(products.isnull().sum().sum()),
        int(aisles.isnull().sum().sum()),
        int(departments.isnull().sum().sum())
    ]
})

display(dinh_dang_bang(bang_missing, "Bảng 4. Tổng số giá trị thiếu theo từng bảng"))

,Bảng,Số giá trị thiếu
0,orders,206209
1,order_products__prior,0
2,order_products__train,0
3,products,0
4,aisles,0
5,departments,0


In [9]:
missing_orders = orders.isnull().sum().reset_index()
missing_orders.columns = ["Cột", "Số giá trị thiếu"]

display(dinh_dang_bang(missing_orders, "Bảng 5. Giá trị thiếu trong bảng orders"))

,Cột,Số giá trị thiếu
0,order_id,0
1,user_id,0
2,eval_set,0
3,order_number,0
4,order_dow,0
5,order_hour_of_day,0
6,days_since_prior_order,206209


### **Nhận xét**
Kết quả cho thấy phần lớn các bảng không có giá trị thiếu.  
Riêng bảng `orders` có dữ liệu thiếu ở cột `days_since_prior_order`. Dữ liệu thiếu này hợp lý, vì những đơn hàng đầu tiên của mỗi khách hàng sẽ không có đơn hàng trước đó để tính khoảng cách thời gian.

## **8. Kiểm tra dòng trùng**
Dòng trùng có thể làm sai lệch thống kê và ảnh hưởng đến kết quả phân tích, vì vậy cần được kiểm tra ngay từ đầu.

In [10]:
bang_trung_toan_dong = pd.DataFrame([
    {"Tên bảng": "orders", "Số dòng trùng hoàn toàn": int(orders.duplicated().sum())},
    {"Tên bảng": "order_products__prior", "Số dòng trùng hoàn toàn": int(order_products_prior.duplicated().sum())},
    {"Tên bảng": "order_products__train", "Số dòng trùng hoàn toàn": int(order_products_train.duplicated().sum())},
    {"Tên bảng": "products", "Số dòng trùng hoàn toàn": int(products.duplicated().sum())},
    {"Tên bảng": "aisles", "Số dòng trùng hoàn toàn": int(aisles.duplicated().sum())},
    {"Tên bảng": "departments", "Số dòng trùng hoàn toàn": int(departments.duplicated().sum())},
])

display(dinh_dang_bang(bang_trung_toan_dong, "Bảng 6. Kiểm tra số dòng trùng hoàn toàn"))

,Tên bảng,Số dòng trùng hoàn toàn
0,orders,0
1,order_products__prior,0
2,order_products__train,0
3,products,0
4,aisles,0
5,departments,0


In [ ]:
bang_trung_theo_khoa = pd.DataFrame([
    {
        "Tên bảng": "orders",
        "Điều kiện kiểm tra": "Trùng order_id",
        "Số dòng trùng": int(orders.duplicated(subset=["order_id"]).sum())
    },
    {
        "Tên bảng": "order_products__prior",
        "Điều kiện kiểm tra": "Trùng cặp (order_id, product_id)",
        "Số dòng trùng": int(order_products_prior.duplicated(subset=["order_id", "product_id"]).sum())
    },
    {
        "Tên bảng": "order_products__train",
        "Điều kiện kiểm tra": "Trùng cặp (order_id, product_id)",
        "Số dòng trùng": int(order_products_train.duplicated(subset=["order_id", "product_id"]).sum())
    },
    {
        "Tên bảng": "products",
        "Điều kiện kiểm tra": "Trùng product_id",
        "Số dòng trùng": int(products.duplicated(subset=["product_id"]).sum())
    },
    {
        "Tên bảng": "aisles",
        "Điều kiện kiểm tra": "Trùng aisle_id",
        "Số dòng trùng": int(aisles.duplicated(subset=["aisle_id"]).sum())
    },
    {
        "Tên bảng": "departments",
        "Điều kiện kiểm tra": "Trùng department_id",
        "Số dòng trùng": int(departments.duplicated(subset=["department_id"]).sum())
    },
])

display(dinh_dang_bang(bang_trung_theo_khoa, "Bảng 7. Kiểm tra trùng theo khóa và quan hệ dữ liệu"))

,Tên bảng,Điều kiện kiểm tra,Số dòng trùng
0,orders,Trùng order_id,0
1,order_products__prior,"Trùng cặp (order_id, product_id)",0
2,order_products__train,"Trùng cặp (order_id, product_id)",0
3,products,Trùng product_id,0
4,aisles,Trùng aisle_id,0
5,departments,Trùng department_id,0


### **Nhận xét**
Kết quả kiểm tra cho thấy các bảng dữ liệu không có dòng trùng hoàn toàn. Đồng thời, khi kiểm tra theo khóa chính và theo cặp quan hệ quan trọng như `order_id`, `product_id` và `(order_id, product_id)`, cũng không phát hiện trùng lặp bất thường. Điều này cho thấy dữ liệu có tính nhất quán tốt và phù hợp để tiếp tục sử dụng trong các bước chuẩn hóa và tích hợp sau đó.

## **9. Thống kê các cột quan trọng và kiểm tra tính duy nhất của khóa**
Bước này nhằm thống kê số lượng giá trị duy nhất của các cột quan trọng trong bộ dữ liệu, đồng thời kiểm tra tính duy nhất và giá trị thiếu của các khóa chính. Qua đó, nhóm có cơ sở đánh giá mức độ nhất quán của dữ liệu trước khi thực hiện các bước chuẩn hóa và tích hợp tiếp theo.

In [ ]:
thong_ke_khoa = pd.DataFrame([
    {"Chỉ tiêu": "Số order_id duy nhất trong orders", "Giá trị": orders["order_id"].nunique()},
    {"Chỉ tiêu": "Số user_id duy nhất trong orders", "Giá trị": orders["user_id"].nunique()},
    {"Chỉ tiêu": "Số order_id duy nhất trong order_products__prior", "Giá trị": order_products_prior["order_id"].nunique()},
    {"Chỉ tiêu": "Số product_id duy nhất trong order_products__prior", "Giá trị": order_products_prior["product_id"].nunique()},
    {"Chỉ tiêu": "Số product_id duy nhất trong products", "Giá trị": products["product_id"].nunique()},
    {"Chỉ tiêu": "Số aisle_id duy nhất trong aisles", "Giá trị": aisles["aisle_id"].nunique()},
    {"Chỉ tiêu": "Số department_id duy nhất trong departments", "Giá trị": departments["department_id"].nunique()},
])

display(dinh_dang_bang(thong_ke_khoa, "Bảng 7. Thống kê số lượng giá trị duy nhất của các cột quan trọng"))

,Chỉ tiêu,Giá trị
0,Số order_id duy nhất trong orders,3421083
1,Số user_id duy nhất trong orders,206209
2,Số order_id duy nhất trong order_products__prior,3214874
3,Số product_id duy nhất trong order_products__prior,49677
4,Số product_id duy nhất trong products,49688
5,Số aisle_id duy nhất trong aisles,134
6,Số department_id duy nhất trong departments,21


In [ ]:
kiem_tra_khoa = pd.DataFrame([
    {
        "Bảng": "orders",
        "Cột khóa": "order_id",
        "Số dòng trùng theo khóa": int(orders.duplicated(subset=["order_id"]).sum()),
        "Số giá trị thiếu": int(orders["order_id"].isnull().sum())
    },
    {
        "Bảng": "products",
        "Cột khóa": "product_id",
        "Số dòng trùng theo khóa": int(products.duplicated(subset=["product_id"]).sum()),
        "Số giá trị thiếu": int(products["product_id"].isnull().sum())
    },
    {
        "Bảng": "aisles",
        "Cột khóa": "aisle_id",
        "Số dòng trùng theo khóa": int(aisles.duplicated(subset=["aisle_id"]).sum()),
        "Số giá trị thiếu": int(aisles["aisle_id"].isnull().sum())
    },
    {
        "Bảng": "departments",
        "Cột khóa": "department_id",
        "Số dòng trùng theo khóa": int(departments.duplicated(subset=["department_id"]).sum()),
        "Số giá trị thiếu": int(departments["department_id"].isnull().sum())
    },
    {
        "Bảng": "order_products__prior",
        "Cột khóa": "(order_id, product_id)",
        "Số dòng trùng theo khóa": int(order_products_prior.duplicated(subset=["order_id", "product_id"]).sum()),
        "Số giá trị thiếu": int(order_products_prior[["order_id", "product_id"]].isnull().sum().sum())
    }
])

display(dinh_dang_bang(kiem_tra_khoa, "Bảng 8. Kiểm tra tính duy nhất của khóa"))

,Bảng,Cột khóa,Số dòng trùng theo khóa,Số giá trị thiếu
0,orders,order_id,0,0
1,products,product_id,0,0
2,aisles,aisle_id,0,0
3,departments,department_id,0,0
4,order_products__prior,"(order_id, product_id)",0,0


### **Nhận xét**
Kết quả thống kê cho thấy bộ dữ liệu có số lượng lớn đơn hàng, người dùng và sản phẩm. Đồng thời, các cột khóa chính như `order_id`, `product_id`, `aisle_id`, `department_id` không bị trùng và không có giá trị thiếu. Riêng bảng `order_products__prior` có thể sử dụng cặp `(order_id, product_id)` như khóa ghép.

## **10. Kiểm tra tính nhất quán của khóa ngoại**
Ngoài khóa chính, cần kiểm tra xem các khóa ngoại giữa các bảng có liên kết đúng với nhau hay không.

In [ ]:
kiem_tra_khoa_ngoai = pd.DataFrame([
    {
        "Mối liên kết": "order_products__prior.order_id -> orders.order_id",
        "Số bản ghi không khớp": int((~order_products_prior["order_id"].isin(orders["order_id"])).sum())
    },
    {
        "Mối liên kết": "order_products__prior.product_id -> products.product_id",
        "Số bản ghi không khớp": int((~order_products_prior["product_id"].isin(products["product_id"])).sum())
    },
    {
        "Mối liên kết": "products.aisle_id -> aisles.aisle_id",
        "Số bản ghi không khớp": int((~products["aisle_id"].isin(aisles["aisle_id"])).sum())
    },
    {
        "Mối liên kết": "products.department_id -> departments.department_id",
        "Số bản ghi không khớp": int((~products["department_id"].isin(departments["department_id"])).sum())
    }
])

display(dinh_dang_bang(kiem_tra_khoa_ngoai, "Bảng 9. Kiểm tra tính nhất quán của khóa ngoại"))

,Mối liên kết,Số bản ghi không khớp
0,order_products__prior.order_id -> orders.order_id,0
1,order_products__prior.product_id -> products.product_id,0
2,products.aisle_id -> aisles.aisle_id,0
3,products.department_id -> departments.department_id,0


### **Nhận xét**
Kết quả kiểm tra cho thấy các khóa ngoại giữa các bảng đều nhất quán.  


## **11. Tổng hợp đánh giá dữ liệu ban đầu**
Sau khi hoàn thành các kiểm tra cơ bản, tiến hành tổng hợp chất lượng dữ liệu của toàn bộ các bảng đầu vào.

In [ ]:
tong_hop_du_lieu_goc = pd.DataFrame([
    tom_tat_bang(orders, "orders"),
    tom_tat_bang(order_products_prior, "order_products__prior"),
    tom_tat_bang(order_products_train, "order_products__train"),
    tom_tat_bang(products, "products"),
    tom_tat_bang(aisles, "aisles"),
    tom_tat_bang(departments, "departments"),
])

display(dinh_dang_bang(tong_hop_du_lieu_goc, "Bảng 10. Tổng hợp chất lượng dữ liệu gốc"))

,Tên bảng,Số dòng,Số cột,Giá trị thiếu,Dòng trùng
0,orders,3421083,7,206209,0
1,order_products__prior,32434489,4,0,0
2,order_products__train,1384617,4,0,0
3,products,49688,4,0,0
4,aisles,134,2,0,0
5,departments,21,2,0,0


## **12. Kết luận Bước 1**
Qua quá trình khởi tạo và khảo sát dữ liệu ban đầu, có thể rút ra một số kết luận như sau:

- Bộ dữ liệu Instacart có quy mô lớn và được tổ chức theo mô hình quan hệ nhiều bảng
- Cấu trúc dữ liệu rõ ràng, thuận lợi cho việc chuẩn hoá và tích hợp
- Dữ liệu hầu như không có lỗi thiếu nghiêm trọng và không có dòng trùng
- Các khóa chính đảm bảo tính duy nhất
- Các khóa ngoại giữa các bảng đều nhất quán
- Cột `days_since_prior_order` có giá trị thiếu nhưng đây là thiếu hợp lệ theo ngữ nghĩa nghiệp vụ

Như vậy, dữ liệu đầu vào đạt chất lượng tốt và sẵn sàng cho bước tiếp theo là **chuẩn hoá dữ liệu**.

## **13. Mục tiêu của bước chuẩn hoá dữ liệu**

Sau khi đã khảo sát dữ liệu ban đầu, bước tiếp theo là chuẩn hoá dữ liệu để đưa dữ liệu về trạng thái gọn hơn, rõ nghĩa hơn và sẵn sàng cho quá trình tích hợp.

Trong bước này, báo cáo tập trung vào các công việc chính sau:

- lựa chọn đúng các bảng và thuộc tính cần thiết,
- giữ lại tập dữ liệu phù hợp với phạm vi bài giữa kì,
- xử lý giá trị thiếu theo đúng ngữ nghĩa nghiệp vụ,
- chuẩn hoá các trường văn bản,
- tối ưu kiểu dữ liệu để giảm bộ nhớ sử dụng,
- xuất ra các tập dữ liệu sạch làm đầu vào cho bước tích hợp.

In [11]:
orders_chon = orders[
    [
        "order_id",
        "user_id",
        "eval_set",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order"
    ]
].copy()

order_products_prior_chon = order_products_prior[
    [
        "order_id",
        "product_id",
        "add_to_cart_order",
        "reordered"
    ]
].copy()

products_chon = products[
    [
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    ]
].copy()

aisles_chon = aisles[
    [
        "aisle_id",
        "aisle"
    ]
].copy()

departments_chon = departments[
    [
        "department_id",
        "department"
    ]
].copy()

## **14. Lựa chọn các bảng và thuộc tính cần thiết**
Ở bước này, nhóm chỉ giữ lại các bảng và các thuộc tính phục vụ trực tiếp cho bài toán, nhằm giảm độ phức tạp của dữ liệu và thuận tiện hơn cho quá trình chuẩn hoá.

In [12]:
bang_lua_chon = pd.DataFrame([
    {"Bảng sau lựa chọn": "orders_chon", "Số dòng": orders_chon.shape[0], "Số cột": orders_chon.shape[1]},
    {"Bảng sau lựa chọn": "order_products_prior_chon", "Số dòng": order_products_prior_chon.shape[0], "Số cột": order_products_prior_chon.shape[1]},
    {"Bảng sau lựa chọn": "products_chon", "Số dòng": products_chon.shape[0], "Số cột": products_chon.shape[1]},
    {"Bảng sau lựa chọn": "aisles_chon", "Số dòng": aisles_chon.shape[0], "Số cột": aisles_chon.shape[1]},
    {"Bảng sau lựa chọn": "departments_chon", "Số dòng": departments_chon.shape[0], "Số cột": departments_chon.shape[1]},
])

display(dinh_dang_bang(bang_lua_chon, "Bảng 11. Kết quả lựa chọn các bảng và thuộc tính cần thiết"))

,Bảng sau lựa chọn,Số dòng,Số cột
0,orders_chon,3421083,7
1,order_products_prior_chon,32434489,4
2,products_chon,49688,4
3,aisles_chon,134,2
4,departments_chon,21,2


### **Nhận xét**
Sau bước lựa chọn, dữ liệu vẫn giữ được đầy đủ những thuộc tính cần thiết cho bài giữa kì, đồng thời loại bỏ được các phần không nằm trong phạm vi xử lý chính.

Việc tách riêng các bảng làm việc như `orders_chon`, `products_chon` hay `order_products_prior_chon` cũng giúp phân biệt rõ giữa dữ liệu gốc và dữ liệu đang được chuẩn hoá, từ đó hạn chế nhầm lẫn trong quá trình thao tác.

## **15. Giữ lại tập lịch sử mua hàng (`prior`)**

Trong bảng `orders`, mỗi đơn hàng được gắn với một nhãn thuộc `eval_set`, gồm `prior`, `train` hoặc `test`.  
Trong ba nhãn của `eval_set`, nhóm chỉ chọn tập `prior` vì đây là phần dữ liệu phản ánh đầy đủ nhất lịch sử mua hàng của người dùng và phù hợp với mục tiêu chuẩn bị dữ liệu cho bài toán khai thác luật kết hợp.  

Tập `train` và `test` được tạo ra chủ yếu để phục vụ bài toán dự đoán trong bộ dữ liệu Instacart gốc, không phải là phần dữ liệu lịch sử chính dùng cho phân tích giỏ hàng. So với `prior`, số lượng đơn trong `train` và `test` cũng ít hơn, nên nếu sử dụng sẽ không phản ánh đầy đủ hành vi mua sắm lặp lại của người dùng.  


In [13]:
orders_prior = orders_chon[orders_chon["eval_set"] == "prior"].copy()
orders_prior = orders_prior.drop(columns=["eval_set"])

bang_loc_prior = pd.DataFrame([
    {"Trạng thái": "Tổng số đơn hàng ban đầu", "Giá trị": orders_chon.shape[0]},
    {"Trạng thái": "Số đơn hàng thuộc tập prior", "Giá trị": orders_prior.shape[0]}
])

display(dinh_dang_bang(bang_loc_prior, "Bảng 12. Kết quả lọc tập đơn hàng lịch sử"))

,Trạng thái,Giá trị
0,Tổng số đơn hàng ban đầu,3421083
1,Số đơn hàng thuộc tập prior,3214874


In [14]:
phan_bo_don_hang = orders_chon["eval_set"].value_counts(dropna=False).reset_index()
phan_bo_don_hang.columns = ["eval_set", "Số lượng"]

display(dinh_dang_bang(phan_bo_don_hang, "Bảng 13. Phân bố đơn hàng theo eval_set"))

,eval_set,Số lượng
0,prior,3214874
1,train,131209
2,test,75000


### **Nhận xét**
Kết quả cho thấy phần lớn đơn hàng trong bộ dữ liệu thuộc tập `prior`. Đây cũng chính là phần dữ liệu phản ánh rõ nhất lịch sử mua sắm của người dùng.

Sau khi lọc, cột `eval_set` không còn cần thiết nữa vì toàn bộ các bản ghi còn lại đều thuộc cùng một nhóm. Việc loại bỏ cột này giúp dữ liệu gọn hơn và tránh mang theo một thuộc tính không còn giá trị phân biệt.

## **16. Xử lý giá trị thiếu của `days_since_prior_order` theo ngữ nghĩa nghiệp vụ**

Ở bước khảo sát dữ liệu ban đầu, cột `days_since_prior_order` là cột duy nhất có giá trị thiếu. Tuy nhiên, đây không phải là lỗi dữ liệu mà là đặc điểm hợp lý của nghiệp vụ, vì đơn hàng đầu tiên của mỗi khách hàng sẽ không có đơn hàng trước đó để tính khoảng cách thời gian.

Vì vậy, ở bước này nhóm không xoá dữ liệu thiếu hay thay thế một cách máy móc. Thay vào đó, nhóm giữ nguyên cột gốc `days_since_prior_order` và tạo thêm hai cột hỗ trợ:

- `is_first_order`: cho biết đơn hàng đó có phải là đơn đầu tiên của người dùng hay không
- `days_since_prior_order_filled`: thay giá trị thiếu bằng `0` để thuận tiện cho các thao tác tính toán và xử lý số học

Cách làm này giúp dữ liệu vừa giữ được ý nghĩa ban đầu, vừa dễ sử dụng hơn trong các bước xử lý tiếp theo.

In [15]:
orders_prior["is_first_order"] = orders_prior["days_since_prior_order"].isna().astype("int8")

orders_prior["days_since_prior_order_filled"] = (
    orders_prior["days_since_prior_order"]
    .fillna(0)
    .astype("int16")
)

In [16]:
bang_xu_ly_missing = pd.DataFrame([
    {"Chỉ tiêu": "Số đơn hàng đầu tiên", "Giá trị": int(orders_prior["is_first_order"].sum())},
    {"Chỉ tiêu": "Số đơn hàng không phải đơn đầu tiên", "Giá trị": int((orders_prior["is_first_order"] == 0).sum())},
    {"Chỉ tiêu": "Số giá trị thiếu còn lại ở cột gốc days_since_prior_order", "Giá trị": int(orders_prior["days_since_prior_order"].isnull().sum())},
    {"Chỉ tiêu": "Số giá trị thiếu ở cột hỗ trợ days_since_prior_order_filled", "Giá trị": int(orders_prior["days_since_prior_order_filled"].isnull().sum())}
])

display(dinh_dang_bang(bang_xu_ly_missing, "Bảng 14. Kết quả xử lý giá trị thiếu của days_since_prior_order"))

,Chỉ tiêu,Giá trị
0,Số đơn hàng đầu tiên,206209
1,Số đơn hàng không phải đơn đầu tiên,3008665
2,Số giá trị thiếu còn lại ở cột gốc days_since_prior_order,206209
3,Số giá trị thiếu ở cột hỗ trợ days_since_prior_order_filled,0


In [ ]:
mau_xu_ly_missing = orders_prior[
    ["days_since_prior_order", "days_since_prior_order_filled", "is_first_order"]
].head(10)

display(dinh_dang_bang(mau_xu_ly_missing, "Bảng 15. Minh hoạ cách chuẩn hoá cột days_since_prior_order"))

,days_since_prior_order,days_since_prior_order_filled,is_first_order
0,nan,0,1
1,15.000000,15,0
2,21.000000,21,0
3,29.000000,29,0
4,28.000000,28,0
5,19.000000,19,0
6,20.000000,20,0
7,14.000000,14,0
8,0.000000,0,0
9,30.000000,30,0


## **17. Chuẩn hoá các trường văn bản**

Các trường văn bản như product_name, aisle và department được chuẩn hóa bằng cách xoá khoảng trắng thừa và chuyển về chữ thường. Bước này giúp dữ liệu nhất quán hơn, tránh trường hợp cùng một nội dung nhưng bị xem là khác nhau do khác biệt về định dạng ký tự.

In [17]:
products_chon["product_name"] = (
    products_chon["product_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

aisles_chon["aisle"] = (
    aisles_chon["aisle"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

departments_chon["department"] = (
    departments_chon["department"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [18]:
display(Markdown("### Mẫu dữ liệu sau khi chuẩn hoá văn bản"))

display(dinh_dang_bang(products_chon.head(5), "Bảng 16. Mẫu dữ liệu bảng products sau chuẩn hoá văn bản"))
display(dinh_dang_bang(aisles_chon.head(5), "Bảng 17. Mẫu dữ liệu bảng aisles sau chuẩn hoá văn bản"))
display(dinh_dang_bang(departments_chon.head(5), "Bảng 18. Mẫu dữ liệu bảng departments sau chuẩn hoá văn bản"))

### Mẫu dữ liệu sau khi chuẩn hoá văn bản

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
4,5,Green Chile Anytime Sauce,5,13


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


### **Nhận xét**
Việc chuẩn hoá ở bước này không làm thay đổi nội dung gốc của dữ liệu mà chỉ giúp các trường văn bản đồng nhất hơn về hình thức. Dù là một thao tác nhỏ, đây vẫn là bước cần thiết để bộ dữ liệu sạch và dễ sử dụng hơn

## **18. Tối ưu kiểu dữ liệu**

Vì bộ dữ liệu Instacart có kích thước lớn, đặc biệt ở bảng chi tiết giao dịch, việc tối ưu kiểu dữ liệu là cần thiết để giảm bộ nhớ sử dụng.

Ở bước này, các cột số được đưa về kiểu dữ liệu nhỏ hơn nhưng vẫn đủ khả năng biểu diễn giá trị thực tế của chúng.  
Cách làm này không làm thay đổi ý nghĩa dữ liệu, nhưng giúp việc xử lý trở nên gọn và hiệu quả hơn.

In [19]:
orders_prior = orders_prior.astype({
    "order_id": "int32",
    "user_id": "int32",
    "order_number": "int16",
    "order_dow": "int8",
    "order_hour_of_day": "int8",
    "is_first_order": "int8",
    "days_since_prior_order_filled": "int16"
})

orders_prior["days_since_prior_order"] = orders_prior["days_since_prior_order"].astype("float32")

order_products_prior_chon = order_products_prior_chon.astype({
    "order_id": "int32",
    "product_id": "int32",
    "add_to_cart_order": "int16",
    "reordered": "int8"
})

products_chon = products_chon.astype({
    "product_id": "int32",
    "aisle_id": "int16",
    "department_id": "int8"
})

aisles_chon = aisles_chon.astype({
    "aisle_id": "int16"
})

departments_chon = departments_chon.astype({
    "department_id": "int8"
})

In [ ]:
bang_kieu_du_lieu_sau_chuan_hoa = pd.DataFrame([
    {
        "Bảng": "orders_prior",
        "Kiểu dữ liệu": ", ".join([f"{cot}: {kieu}" for cot, kieu in orders_prior.dtypes.items()])
    },
    {
        "Bảng": "order_products_prior_chon",
        "Kiểu dữ liệu": ", ".join([f"{cot}: {kieu}" for cot, kieu in order_products_prior_chon.dtypes.items()])
    },
    {
        "Bảng": "products_chon",
        "Kiểu dữ liệu": ", ".join([f"{cot}: {kieu}" for cot, kieu in products_chon.dtypes.items()])
    }
])

display(dinh_dang_bang(bang_kieu_du_lieu_sau_chuan_hoa, "Bảng 19. Kiểu dữ liệu sau khi tối ưu"))

,Bảng,Kiểu dữ liệu
0,orders_prior,"order_id: int32, user_id: int32, order_number: int16, order_dow: int8, order_hour_of_day: int8, days_since_prior_order: float32, is_first_order: int8, days_since_prior_order_filled: int16"
1,order_products_prior_chon,"order_id: int32, product_id: int32, add_to_cart_order: int16, reordered: int8"
2,products_chon,"product_id: int32, product_name: string, aisle_id: int16, department_id: int8"


### **Nhận xét**
Sau khi tối ưu, dữ liệu vẫn giữ nguyên giá trị nhưng sử dụng bộ nhớ hợp lý hơn.  
Đây là một bước quan trọng đối với những bộ dữ liệu lớn như Instacart, vì nếu giữ toàn bộ cột số ở kiểu mặc định quá lớn, quá trình xử lý và tích hợp dữ liệu về sau sẽ tốn tài nguyên hơn mức cần thiết.

## **19. Kiểm tra lại dữ liệu sau chuẩn hoá**

Sau khi hoàn tất các bước chuẩn hoá, cần kiểm tra lại dữ liệu một lần nữa để xác nhận rằng:

- số dòng không bị thay đổi ngoài ý muốn,
- dữ liệu không phát sinh thêm lỗi thiếu hoặc trùng,
- các bảng sau chuẩn hoá vẫn đảm bảo tính ổn định trước khi bước sang giai đoạn tích hợp.

In [ ]:
tong_hop_sau_chuan_hoa = pd.DataFrame([
    tom_tat_bang(orders_prior, "orders_prior"),
    tom_tat_bang(order_products_prior_chon, "order_products_prior_chon"),
    tom_tat_bang(products_chon, "products_chon"),
    tom_tat_bang(aisles_chon, "aisles_chon"),
    tom_tat_bang(departments_chon, "departments_chon")
])

display(dinh_dang_bang(tong_hop_sau_chuan_hoa, "Bảng 20. Tổng hợp chất lượng dữ liệu sau chuẩn hoá"))

,Tên bảng,Số dòng,Số cột,Giá trị thiếu,Dòng trùng
0,orders_prior,3214874,8,206209,0
1,order_products_prior_chon,32434489,4,0,0
2,products_chon,49688,4,0,0
3,aisles_chon,134,2,0,0
4,departments_chon,21,2,0,0


In [ ]:
kiem_tra_missing_orders_prior = orders_prior.isnull().sum().reset_index()
kiem_tra_missing_orders_prior.columns = ["Cột", "Số giá trị thiếu"]

display(dinh_dang_bang(kiem_tra_missing_orders_prior, "Bảng 21. Giá trị thiếu trong orders_prior sau chuẩn hoá"))

,Cột,Số giá trị thiếu
0,order_id,0
1,user_id,0
2,order_number,0
3,order_dow,0
4,order_hour_of_day,0
5,days_since_prior_order,206209
6,is_first_order,0
7,days_since_prior_order_filled,0


### **Nhận xét**
Kết quả kiểm tra lại cho thấy dữ liệu sau chuẩn hoá vẫn ổn định.  
Giá trị thiếu chỉ còn xuất hiện ở cột gốc `days_since_prior_order`, và đây vẫn là giá trị thiếu hợp lệ theo ngữ nghĩa nghiệp vụ như đã phân tích trước đó. Ngoài điểm này, các bảng dữ liệu đều ở trạng thái sạch và sẵn sàng cho bước tích hợp.

## **20. Xuất các tập dữ liệu sạch**

Sau khi hoàn thành bước chuẩn hoá, các bảng dữ liệu được lưu lại thành các file riêng.  
Việc tách thành các file sạch giúp quá trình làm việc ở bước tiếp theo rõ ràng hơn, đồng thời cũng tạo ra bộ dữ liệu trung gian có thể tái sử dụng khi cần.

In [20]:
orders_prior.to_csv("clean_orders_prior.csv", index=False)
order_products_prior_chon.to_csv("clean_order_products_prior.csv", index=False)
products_chon.to_csv("clean_products.csv", index=False)
aisles_chon.to_csv("clean_aisles.csv", index=False)
departments_chon.to_csv("clean_departments.csv", index=False)

bang_output_clean = pd.DataFrame([
    {"Tên file đầu ra": "clean_orders_prior.csv", "Số dòng": orders_prior.shape[0], "Số cột": orders_prior.shape[1]},
    {"Tên file đầu ra": "clean_order_products_prior.csv", "Số dòng": order_products_prior_chon.shape[0], "Số cột": order_products_prior_chon.shape[1]},
    {"Tên file đầu ra": "clean_products.csv", "Số dòng": products_chon.shape[0], "Số cột": products_chon.shape[1]},
    {"Tên file đầu ra": "clean_aisles.csv", "Số dòng": aisles_chon.shape[0], "Số cột": aisles_chon.shape[1]},
    {"Tên file đầu ra": "clean_departments.csv", "Số dòng": departments_chon.shape[0], "Số cột": departments_chon.shape[1]},
])

display(dinh_dang_bang(bang_output_clean, "Bảng 22. Các tập dữ liệu sạch sau bước chuẩn hoá"))

,Tên file đầu ra,Số dòng,Số cột
0,clean_orders_prior.csv,3214874,8
1,clean_order_products_prior.csv,32434489,4
2,clean_products.csv,49688,4
3,clean_aisles.csv,134,2
4,clean_departments.csv,21,2


## **21. Kết luận Bước 2**

Qua bước chuẩn hoá, dữ liệu đã được rút gọn về đúng phạm vi cần thiết cho bài giữa kì. Tập `prior` được giữ lại làm dữ liệu chính vì phản ánh lịch sử mua hàng thực tế với quy mô lớn nhất. Các giá trị thiếu được xử lý theo đúng ngữ nghĩa nghiệp vụ, các trường văn bản được làm nhất quán hơn, và kiểu dữ liệu cũng được tối ưu để phù hợp với kích thước lớn của bộ dữ liệu.

Kết quả của bước này là một bộ dữ liệu sạch, gọn và ổn định hơn, gồm các file:

- `clean_orders_prior.csv`
- `clean_order_products_prior.csv`
- `clean_products.csv`
- `clean_aisles.csv`
- `clean_departments.csv`

Đây sẽ là đầu vào trực tiếp cho bước tiếp theo là **tích hợp dữ liệu sau chuẩn hoá**.

## **22. Tích hợp dữ liệu theo hướng tiết kiệm tài nguyên**

Sau khi dữ liệu đã được chuẩn hoá, bước tiếp theo là tích hợp các bảng liên quan để tạo tập dữ liệu phục vụ phân tích luật kết hợp. Tuy nhiên, do bộ dữ liệu Instacart có kích thước lớn và cấu hình máy không cho phép merge toàn bộ dữ liệu gốc một cách hiệu quả, quá trình tích hợp được thực hiện theo hướng rút gọn trước khi ghép.

Cách làm này nhằm giảm khối lượng xử lý nhưng vẫn giữ lại các thành phần cần thiết của bài toán như giao dịch, sản phẩm và thông tin phân loại. Trong phạm vi đồ án môn học, tập dữ liệu sau rút gọn vẫn phù hợp để tiếp tục chuẩn bị cho bước khai thác luật kết hợp.

## **23. Nguyên tắc tích hợp dữ liệu tối ưu bộ nhớ**

Quá trình tích hợp được thực hiện theo đúng quan hệ giữa các bảng trong bộ dữ liệu Instacart, nhưng có thêm bước rút gọn quy mô trước khi ghép:

- lấy mẫu một phần `user_id` từ `orders_prior`,
- lấy các `order_id` tương ứng,
- lọc `order_products_prior_chon` theo tập `order_id` này,
- tiếp tục giữ lại các `product_id` phổ biến nhất,
- sau đó mới ghép với `orders_prior`, `products_chon`, `aisles_chon`, `departments_chon`.

Thứ tự này giúp giảm đáng kể dung lượng dữ liệu cần xử lý trước khi merge, đồng thời vẫn giữ được cấu trúc cần thiết cho các bước phân tích tiếp theo.


## **24. Tạo tập mẫu giao dịch trước khi ghép**

Ở bước này, dữ liệu được lấy mẫu theo `user_id` từ tập `orders_prior` để giảm khối lượng xử lý trước khi ghép. Cách lấy mẫu theo người dùng giúp giữ được cấu trúc giao dịch cần thiết cho các bước sau.


In [23]:
# ==============================
# CẤU HÌNH LẤY MẪU GIAO DỊCH
# ==============================
SO_USER_MAU = 5000         
RANDOM_STATE = 42

so_user_co_san = orders_prior["user_id"].nunique()
so_user_lay_mau = min(SO_USER_MAU, so_user_co_san)

user_mau = (
    orders_prior["user_id"]
    .drop_duplicates()
    .sample(n=so_user_lay_mau, random_state=RANDOM_STATE)
)

orders_prior_mau = orders_prior[
    orders_prior["user_id"].isin(user_mau)
].copy()

order_ids_mau = orders_prior_mau["order_id"].unique()

order_products_prior_mau = order_products_prior_chon[
    order_products_prior_chon["order_id"].isin(order_ids_mau)
].copy()

bang_mau_giao_dich = pd.DataFrame([
    {"Chỉ tiêu": "Số user_id trong orders_prior gốc", "Giá trị": int(so_user_co_san)},
    {"Chỉ tiêu": "Số user_id được lấy mẫu", "Giá trị": int(so_user_lay_mau)},
    {"Chỉ tiêu": "Số order_id trong tập mẫu", "Giá trị": int(orders_prior_mau["order_id"].nunique())},
    {"Chỉ tiêu": "Số dòng chi tiết sản phẩm trong tập mẫu", "Giá trị": int(order_products_prior_mau.shape[0])}
])

display(dinh_dang_bang(bang_mau_giao_dich, "Bảng 23. Kết quả lấy mẫu giao dịch trước khi ghép"))


,Chỉ tiêu,Giá trị
0,Số user_id trong orders_prior gốc,206209
1,Số user_id được lấy mẫu,5000
2,Số order_id trong tập mẫu,77862
3,Số dòng chi tiết sản phẩm trong tập mẫu,778814


In [ ]:
display(Markdown("### Mẫu dữ liệu sau khi lấy subset theo user_id"))
display(orders_prior_mau.head())
display(order_products_prior_mau.head())


### Mẫu dữ liệu sau khi lấy subset theo user_id

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,is_first_order,days_since_prior_order_filled
1074,364423,71,1,3,10,NaN,1,0
1075,1962560,71,2,3,16,7.0,0,7
1076,1841185,71,3,5,12,9.0,0,9
1077,3112384,71,4,0,19,2.0,0,2
1078,2012622,71,5,4,21,18.0,0,18


,order_id,product_id,add_to_cart_order,reordered
268,28,35108,1,0
269,28,40593,2,1
270,28,17461,3,0
271,28,22825,4,1
272,28,25256,5,1


## **25. Phân tích độ bao phủ và áp dụng ngưỡng lọc sản phẩm**

Sau khi có tập mẫu giao dịch, dữ liệu được đánh giá ở nhiều mức Top N sản phẩm khác nhau để chọn ngưỡng lọc phù hợp dựa trên độ bao phủ. Từ kết quả này, ngưỡng được chọn sẽ được áp dụng để tạo tập dữ liệu dùng cho các bước tiếp theo.


In [24]:
def danh_gia_top_n(df, ds_top_n):
    ket_qua = []

    tong_dong = len(df)
    tong_order = df["order_id"].nunique()
    tong_product = df["product_id"].nunique()

    tan_suat = (
        df["product_id"]
        .value_counts()
        .rename_axis("product_id")
        .reset_index(name="so_lan_xuat_hien")
    )

    for top_n in ds_top_n:
        product_giu = tan_suat.head(min(top_n, tan_suat.shape[0]))["product_id"]

        df_loc = df[
            df["product_id"].isin(product_giu)
        ].drop_duplicates(subset=["order_id", "product_id"]).copy()

        ket_qua.append({
            "Top N": int(top_n),
            "Số product_id giữ lại": int(df_loc["product_id"].nunique()),
            "Số dòng sau lọc": int(len(df_loc)),
            "Độ bao phủ theo dòng (%)": round(len(df_loc) / tong_dong * 100, 2),
            "Số order_id còn lại": int(df_loc["order_id"].nunique()),
            "Độ bao phủ theo đơn hàng (%)": round(df_loc["order_id"].nunique() / tong_order * 100, 2),
            "Tỷ lệ loại sản phẩm giữ lại (%)": round(df_loc["product_id"].nunique() / tong_product * 100, 2)
        })

    return pd.DataFrame(ket_qua)

ds_top_n_thu = [800, 1000, 1500, 2000, 3000, 5000]

bang_so_sanh_top_n = danh_gia_top_n(
    order_products_prior_mau,
    ds_top_n=ds_top_n_thu
)

display(dinh_dang_bang(
    bang_so_sanh_top_n,
    "Bảng 24. Phân tích độ bao phủ theo các mức Top N sản phẩm"
))


,Top N,Số product_id giữ lại,Số dòng sau lọc,Độ bao phủ theo dòng (%),Số order_id còn lại,Độ bao phủ theo đơn hàng (%),Tỷ lệ loại sản phẩm giữ lại (%)
0,800,800,396229,50.880000,70511,90.560000,2.760000
1,1000,1000,425911,54.690000,71466,91.790000,3.450000
2,1500,1500,481906,61.880000,73373,94.230000,5.180000
3,2000,2000,522417,67.080000,74330,95.460000,6.900000
4,3000,3000,579185,74.370000,75519,96.990000,10.360000
5,5000,5000,647118,83.090000,76595,98.370000,17.260000


Từ kết quả phân tích độ bao phủ ở trên, nhóm chọn giữ lại Top 3000 sản phẩm để lọc dữ liệu trên tập mẫu giao dịch. 

In [25]:
TOP_N_SAN_PHAM = 3000

tan_suat_san_pham = (
    order_products_prior_mau["product_id"]
    .value_counts()
    .rename_axis("product_id")
    .reset_index(name="so_lan_xuat_hien")
)

san_pham_duoc_giu = tan_suat_san_pham.head(
    min(TOP_N_SAN_PHAM, tan_suat_san_pham.shape[0])
).copy()

order_products_prior_loc = order_products_prior_mau[
    order_products_prior_mau["product_id"].isin(san_pham_duoc_giu["product_id"])
].drop_duplicates(subset=["order_id", "product_id"]).copy()

bang_ghep_2 = pd.DataFrame([
    {"Chỉ tiêu": "Số product_id khác nhau trong tập mẫu", "Giá trị": int(order_products_prior_mau["product_id"].nunique())},
    {"Chỉ tiêu": f"Số product_id được giữ lại (Top {TOP_N_SAN_PHAM})", "Giá trị": int(san_pham_duoc_giu["product_id"].nunique())},
    {"Chỉ tiêu": "Số dòng còn lại sau lọc product_id", "Giá trị": int(order_products_prior_loc.shape[0])},
    {"Chỉ tiêu": "Số order_id còn lại sau lọc", "Giá trị": int(order_products_prior_loc["order_id"].nunique())}
])

display(dinh_dang_bang(
    bang_ghep_2,
    "Bảng 25. Kết quả lọc sản phẩm theo ngưỡng đã chọn"
))

,Chỉ tiêu,Giá trị
0,Số product_id khác nhau trong tập mẫu,28966
1,Số product_id được giữ lại (Top 3000),3000
2,Số dòng còn lại sau lọc product_id,579185
3,Số order_id còn lại sau lọc,75519


In [26]:
display(dinh_dang_bang(
    tan_suat_san_pham.head(10),
    "Bảng 26. Một số product_id có tần suất xuất hiện cao trong tập mẫu"
))


,product_id,so_lan_xuat_hien
0,24852,10866
1,13176,9201
2,21137,6572
3,21903,5881
4,47209,5270
5,47766,4498
6,26209,3713
7,47626,3609
8,27845,3509
9,16797,3502


**Nhận xét:**  

Kết quả phân tích cho thấy mức Top 3000 vẫn giữ được độ bao phủ cao theo số dòng giao dịch và số đơn hàng, đồng thời số lượng sản phẩm còn lại vẫn phù hợp với khả năng xử lý của máy. Sau khi áp dụng ngưỡng này, dữ liệu được rút gọn hơn để tiếp tục bước tích hợp.

## **26. Ghép tập mẫu đã lọc với thông tin đơn hàng và sản phẩm**

Sau khi đã có tập chi tiết giao dịch gọn hơn, dữ liệu mới được ghép với các bảng mô tả để bổ sung thông tin người dùng, thời gian mua, tên sản phẩm, quầy hàng và ngành hàng. Việc ghép ở giai đoạn này nhẹ hơn nhiều so với ghép toàn bộ dữ liệu ngay từ đầu.


In [27]:
du_lieu_giao_dich = order_products_prior_loc.merge(
    orders_prior_mau,
    on="order_id",
    how="inner",
    validate="many_to_one"
)

du_lieu_san_pham = du_lieu_giao_dich.merge(
    products_chon[["product_id", "product_name", "aisle_id", "department_id"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

du_lieu_aisle = du_lieu_san_pham.merge(
    aisles_chon,
    on="aisle_id",
    how="left",
    validate="many_to_one"
)

du_lieu_tich_hop = du_lieu_aisle.merge(
    departments_chon,
    on="department_id",
    how="left",
    validate="many_to_one"
)


In [28]:
bang_ghep_3 = pd.DataFrame([
    {"Chỉ tiêu": "Số dòng sau khi ghép với orders_prior_mau", "Giá trị": int(du_lieu_giao_dich.shape[0])},
    {"Chỉ tiêu": "Số cột sau khi ghép với orders_prior_mau", "Giá trị": int(du_lieu_giao_dich.shape[1])},
    {"Chỉ tiêu": "Số dòng sau khi ghép thêm products/aisles/departments", "Giá trị": int(du_lieu_tich_hop.shape[0])},
    {"Chỉ tiêu": "Số cột của bảng tích hợp rút gọn", "Giá trị": int(du_lieu_tich_hop.shape[1])}
])

display(dinh_dang_bang(bang_ghep_3, "Bảng 28. Kết quả ghép trên tập dữ liệu rút gọn"))


,Chỉ tiêu,Giá trị
0,Số dòng sau khi ghép với orders_prior_mau,579185
1,Số cột sau khi ghép với orders_prior_mau,11
2,Số dòng sau khi ghép thêm products/aisles/departments,579185
3,Số cột của bảng tích hợp rút gọn,16


In [29]:
kiem_tra_phan_loai_sau_ghep = pd.DataFrame([
    {"Cột kiểm tra": "product_name", "Số giá trị thiếu": int(du_lieu_tich_hop["product_name"].isnull().sum())},
    {"Cột kiểm tra": "aisle", "Số giá trị thiếu": int(du_lieu_tich_hop["aisle"].isnull().sum())},
    {"Cột kiểm tra": "department", "Số giá trị thiếu": int(du_lieu_tich_hop["department"].isnull().sum())}
])

display(dinh_dang_bang(kiem_tra_phan_loai_sau_ghep, "Bảng 27. Kiểm tra dữ liệu thiếu sau khi ghép"))


,Cột kiểm tra,Số giá trị thiếu
0,product_name,0
1,aisle,0
2,department,0


### **Nhận xét**
Sau khi ghép thêm các bảng mô tả, dữ liệu đã đủ ngữ nghĩa để trình bày và phân tích, nhưng kích thước vẫn được kiểm soát vì toàn bộ quá trình chỉ diễn ra trên tập mẫu đã rút gọn.


## **27. Chuẩn hoá lại tên cột và sắp xếp cấu trúc bảng cuối cùng**

Sau quá trình ghép bảng, một số cột mô tả như `aisle` và `department` được đổi tên để rõ nghĩa hơn trong bảng tổng hợp cuối cùng.  
Đồng thời, các cột cũng được sắp xếp lại theo nhóm thông tin để bảng dữ liệu dễ đọc và dễ sử dụng hơn.

In [30]:
du_lieu_tich_hop = du_lieu_tich_hop.rename(columns={
    "aisle": "aisle_name",
    "department": "department_name"
})

du_lieu_tich_hop = du_lieu_tich_hop[
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "days_since_prior_order",
        "days_since_prior_order_filled",
        "is_first_order",
        "product_id",
        "product_name",
        "aisle_id",
        "aisle_name",
        "department_id",
        "department_name",
        "add_to_cart_order",
        "reordered"
    ]
]

display(Markdown("### Mẫu dữ liệu sau khi tích hợp hoàn chỉnh"))
display(du_lieu_tich_hop.head())

### Mẫu dữ liệu sau khi tích hợp hoàn chỉnh

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_filled,is_first_order,product_id,product_name,aisle_id,aisle_name,department_id,department_name,add_to_cart_order,reordered
0,28,98256,29,3,13,6.0,6,0,35108,Salted Butter,36,butter,16,dairy eggs,1,0
1,28,98256,29,3,13,6.0,6,0,40593,Cream Cheese,108,other creams cheeses,16,dairy eggs,2,1
2,28,98256,29,3,13,6.0,6,0,17461,Air Chilled Organic Boneless Skinless Chicken ...,35,poultry counter,12,meat seafood,3,0
3,28,98256,29,3,13,6.0,6,0,22825,Organic D'Anjou Pears,24,fresh fruits,4,produce,4,1
4,28,98256,29,3,13,6.0,6,0,25256,Cultured Low Fat Buttermilk,84,milk,16,dairy eggs,5,1


### **Nhận xét**
Sau khi chuẩn hoá tên cột và sắp xếp lại cấu trúc, bảng dữ liệu tổng hợp rút gọn trở nên rõ ràng và dễ sử dụng hơn. Đây là bảng trung tâm để tiếp tục tạo dữ liệu đầu vào cho luật kết hợp trên máy cá nhân.


## **28. Kiểm tra lại chất lượng dữ liệu sau tích hợp rút gọn**

Sau khi hoàn tất quá trình ghép trên tập mẫu, cần kiểm tra lại chất lượng của tập dữ liệu đầu ra để chắc chắn rằng:

- không bị mất dòng ngoài ý muốn,
- không phát sinh dòng trùng,
- không xuất hiện lỗi thiếu ở các cột mới được ghép,
- dữ liệu vẫn giữ được tính nhất quán sau toàn bộ quá trình tích hợp.


In [31]:
tong_hop_sau_tich_hop = pd.DataFrame([
    {"Chỉ tiêu": "Số dòng của bảng tích hợp rút gọn", "Giá trị": int(du_lieu_tich_hop.shape[0])},
    {"Chỉ tiêu": "Số cột của bảng tích hợp rút gọn", "Giá trị": int(du_lieu_tich_hop.shape[1])},
    {"Chỉ tiêu": "Số dòng trùng", "Giá trị": int(du_lieu_tich_hop.duplicated().sum())},
    {"Chỉ tiêu": "Số order_id duy nhất", "Giá trị": int(du_lieu_tich_hop["order_id"].nunique())},
    {"Chỉ tiêu": "Số user_id duy nhất", "Giá trị": int(du_lieu_tich_hop["user_id"].nunique())},
    {"Chỉ tiêu": "Số product_id duy nhất", "Giá trị": int(du_lieu_tich_hop["product_id"].nunique())}
])

display(dinh_dang_bang(tong_hop_sau_tich_hop, "Bảng 28. Tổng quan dữ liệu sau tích hợp rút gọn"))


,Chỉ tiêu,Giá trị
0,Số dòng của bảng tích hợp rút gọn,579185
1,Số cột của bảng tích hợp rút gọn,16
2,Số dòng trùng,0
3,Số order_id duy nhất,75519
4,Số user_id duy nhất,4993
5,Số product_id duy nhất,3000


In [ ]:
missing_sau_tich_hop = du_lieu_tich_hop.isnull().sum().reset_index()
missing_sau_tich_hop.columns = ["Cột", "Số giá trị thiếu"]

display(dinh_dang_bang(missing_sau_tich_hop, "Bảng 29. Kiểm tra giá trị thiếu sau tích hợp rút gọn"))


,Cột,Số giá trị thiếu
0,order_id,0
1,user_id,0
2,order_number,0
3,order_dow,0
4,order_hour_of_day,0
5,days_since_prior_order,35428
6,days_since_prior_order_filled,0
7,is_first_order,0
8,product_id,0
9,product_name,0


## **29. Xuất bảng dữ liệu tổng hợp rút gọn**

Sau khi hoàn tất bước tích hợp, bảng dữ liệu tổng hợp rút gọn được lưu ra file để dùng cho các bước chuẩn bị dữ liệu đầu vào cho luật kết hợp. Ở phiên bản tối ưu này, không còn xuất `instacart_merged_full.csv` trên toàn bộ dữ liệu nữa.


In [32]:
ten_file_subset = "instacart_subset_for_rules.csv"
du_lieu_tich_hop.to_csv(ten_file_subset, index=False)

bang_output_tich_hop = pd.DataFrame([
    {
        "Tên file đầu ra": ten_file_subset,
        "Số dòng": int(du_lieu_tich_hop.shape[0]),
        "Số cột": int(du_lieu_tich_hop.shape[1])
    }
])

display(dinh_dang_bang(bang_output_tich_hop, "Bảng 30. File dữ liệu tổng hợp rút gọn sau bước tích hợp"))


,Tên file đầu ra,Số dòng,Số cột
0,instacart_subset_for_rules.csv,579185,16


## **30. Kết luận Bước 3**

Bước tích hợp dữ liệu vẫn được thực hiện đầy đủ, nhưng đã được thiết kế lại theo hướng phù hợp hơn với máy cá nhân. Thay vì ghép toàn bộ dữ liệu rồi mới giảm tải, báo cáo lấy mẫu giao dịch trước, lọc các sản phẩm phổ biến và chỉ sau đó mới tích hợp thông tin mô tả.

Kết quả thu được là file `instacart_subset_for_rules.csv`, đóng vai trò là tập dữ liệu đầu vào thực tế cho phần chuẩn bị luật kết hợp. Cách làm này giúp giữ lại bộ dữ liệu Instacart mà không làm notebook bị quá tải.


## **35. Mục tiêu của bước chuẩn bị dữ liệu đầu vào cho luật kết hợp**

Sau khi đã hoàn thành quá trình chuẩn bị dữ liệu, chuẩn hoá dữ liệu và tích hợp dữ liệu rút gọn, bước tiếp theo là chuyển dữ liệu sang dạng phù hợp để có thể sử dụng cho các thuật toán khai thác tập phổ biến và luật kết hợp ở giai đoạn sau.

Trong các bài toán luật kết hợp, dữ liệu đầu vào thường không được sử dụng trực tiếp dưới dạng bảng chi tiết dòng-sản-phẩm như `instacart_subset_for_rules.csv`. Thay vào đó, dữ liệu cần được biến đổi sang dạng:

- mỗi dòng đại diện cho một giao dịch,
- mỗi cột đại diện cho một sản phẩm,
- giá trị trong ô cho biết sản phẩm đó có xuất hiện trong giao dịch hay không.

Vì vậy, ở bước này, báo cáo thực hiện ba công việc chính:

1. kiểm tra và lọc nhẹ lại tập sản phẩm trên dữ liệu rút gọn,
2. xây dựng tập giao dịch theo `order_id`,
3. tạo ma trận giao dịch ở dạng sparse để làm đầu vào cho phần cuối kì.


## **36. Kiểm tra lại tần suất sản phẩm trên dữ liệu rút gọn**

Do dữ liệu đã được rút gọn ở bước trước, việc lọc ở bước này chỉ mang tính tinh chỉnh thêm để đảm bảo các sản phẩm được giữ lại vẫn xuất hiện đủ thường xuyên trong tập dữ liệu cuối cùng dùng cho luật kết hợp.


In [33]:
# Có thể điều chỉnh ngưỡng này tuỳ vào máy và mục tiêu phân tích
nguong_xuat_hien = 20

tan_suat_san_pham_cuoi = (
    du_lieu_tich_hop["product_id"]
    .value_counts()
    .rename_axis("product_id")
    .reset_index(name="so_lan_xuat_hien")
)

san_pham_duoc_giu = tan_suat_san_pham_cuoi[
    tan_suat_san_pham_cuoi["so_lan_xuat_hien"] >= nguong_xuat_hien
].copy()

bang_loc_san_pham = pd.DataFrame([
    {"Chỉ tiêu": "Tổng số product_id trong dữ liệu rút gọn", "Giá trị": int(du_lieu_tich_hop["product_id"].nunique())},
    {"Chỉ tiêu": f"Số product_id có tần suất >= {nguong_xuat_hien}", "Giá trị": int(san_pham_duoc_giu["product_id"].nunique())}
])

display(dinh_dang_bang(bang_loc_san_pham, "Bảng 32. Kết quả lọc sản phẩm theo tần suất xuất hiện cuối cùng"))


,Chỉ tiêu,Giá trị
0,Tổng số product_id trong dữ liệu rút gọn,3000
1,Số product_id có tần suất >= 20,3000


In [34]:
tan_suat_hien_thi = (
    tan_suat_san_pham_cuoi
    .merge(products_chon[["product_id", "product_name"]], on="product_id", how="left")
    [["product_id", "product_name", "so_lan_xuat_hien"]]
)

display(dinh_dang_bang(
    tan_suat_hien_thi.head(10),
    "Bảng 33. Một số sản phẩm có tần suất xuất hiện cao trong dữ liệu rút gọn"
))


,product_id,product_name,so_lan_xuat_hien
0,24852,Banana,10866
1,13176,Bag of Organic Bananas,9201
2,21137,Organic Strawberries,6572
3,21903,Organic Baby Spinach,5881
4,47209,Organic Hass Avocado,5270
5,47766,Organic Avocado,4498
6,26209,Limes,3713
7,47626,Large Lemon,3609
8,27845,Organic Whole Milk,3509
9,16797,Strawberries,3502


### **Nhận xét**
Ngưỡng lọc ở giai đoạn này giúp tinh gọn thêm tập sản phẩm trước khi tạo ma trận giao dịch. Vì dữ liệu đã được rút gọn từ trước, ngưỡng lọc có thể đặt thấp hơn so với dữ liệu full.


## **37. Tạo tập dữ liệu đã lọc để chuẩn bị xây dựng giao dịch**

Sau khi xác định được danh sách sản phẩm cần giữ lại, bước tiếp theo là lọc bảng `du_lieu_tich_hop` để chỉ còn các bản ghi chứa những sản phẩm này.


In [35]:
du_lieu_loc = du_lieu_tich_hop[
    du_lieu_tich_hop["product_id"].isin(san_pham_duoc_giu["product_id"])
].copy()

bang_du_lieu_loc = pd.DataFrame([
    {"Chỉ tiêu": "Số dòng trong du_lieu_tich_hop", "Giá trị": int(du_lieu_tich_hop.shape[0])},
    {"Chỉ tiêu": "Số dòng sau khi lọc sản phẩm", "Giá trị": int(du_lieu_loc.shape[0])},
    {"Chỉ tiêu": "Số transaction còn lại", "Giá trị": int(du_lieu_loc["order_id"].nunique())},
    {"Chỉ tiêu": "Số product_id còn lại", "Giá trị": int(du_lieu_loc["product_id"].nunique())}
])

display(dinh_dang_bang(bang_du_lieu_loc, "Bảng 34. Quy mô dữ liệu sau khi lọc sản phẩm"))


,Chỉ tiêu,Giá trị
0,Số dòng trong du_lieu_tich_hop,579185
1,Số dòng sau khi lọc sản phẩm,579185
2,Số transaction còn lại,75519
3,Số product_id còn lại,3000


### **Nhận xét**
Sau bước lọc cuối, dữ liệu đã đủ gọn để chuyển sang dạng giao dịch. Đây là quy mô phù hợp hơn nhiều cho các bước khai thác luật kết hợp trên máy cá nhân.


## **38. Xây dựng tập giao dịch theo đơn hàng**

Trong bài toán luật kết hợp, một giao dịch thường được hiểu là một đơn hàng. Ở phiên bản tối ưu này, dữ liệu giao dịch được gom theo `order_id` nhưng lưu danh sách `product_id` để giảm nhẹ bộ nhớ. Tên sản phẩm sẽ chỉ được dùng ở bước hiển thị và diễn giải kết quả.


In [36]:
basket_data = (
    du_lieu_loc.groupby("order_id")["product_id"]
    .apply(list)
    .reset_index()
)

basket_data.columns = ["order_id", "danh_sach_product_id"]

bang_basket = pd.DataFrame([
    {"Chỉ tiêu": "Số giao dịch trong basket_data", "Giá trị": int(basket_data.shape[0])},
    {"Chỉ tiêu": "Số cột của basket_data", "Giá trị": int(basket_data.shape[1])}
])

display(dinh_dang_bang(bang_basket, "Bảng 35. Tổng quan tập giao dịch sau khi gom theo order_id"))


,Chỉ tiêu,Giá trị
0,Số giao dịch trong basket_data,75519
1,Số cột của basket_data,2


In [37]:
mapping_ten_san_pham = (
    du_lieu_loc[["product_id", "product_name"]]
    .drop_duplicates()
    .set_index("product_id")["product_name"]
    .to_dict()
)

basket_data_hien_thi = basket_data.head(10).copy()
basket_data_hien_thi["danh_sach_san_pham"] = basket_data_hien_thi["danh_sach_product_id"].apply(
    lambda ds: ", ".join(mapping_ten_san_pham.get(x, str(x)) for x in ds)
)

basket_data_hien_thi = basket_data_hien_thi[["order_id", "danh_sach_san_pham"]]

display(dinh_dang_bang(
    basket_data_hien_thi,
    "Bảng 36. Minh hoạ một số giao dịch sau khi gom sản phẩm theo order_id"
))


,order_id,danh_sach_san_pham
0,28,"Salted Butter, Cream Cheese, Air Chilled Organic Boneless Skinless Chicken Breasts, Organic D'Anjou Pears, Cultured Low Fat Buttermilk, Large Lemon, Organic Strawberry Fruit Spread, Large Greenhouse Tomato, Original Semisoft Cheese, Whole Organic Omega 3 Milk, Organic Heavy Whipping Cream, Organic Hass Avocado, 1% Lowfat Milk, Organic Whole Milk Strawberry Beet Berry Yogurt Pouch, Organic Black Plum, Organic Red Delicious Apple"
1,67,"Bag of Organic Bananas, Plain Greek Yogurt, Maple Glazed Honey Ham, Organic Grade A Large Brown Eggs, Cherry Garcia Ice Cream, Prosciutto, Thick & Crispy Tortilla Chips, High Pulp Orange Juice"
2,71,"Tilapia Filet, Toasted Coconut Chips Original Recipe, Organic Hass Avocado, Organic Crushed Fire Roasted Tomatoes, Organic Lemon, Fresh Ginger Root, Organic Cilantro, Limes, Sparkling Water Grapefruit"
3,109,"Original Orange Juice, Organic Unsalted Butter, Frozen Organic Strawberries, Electrolyte Enhanced Water, Rich & Hearty Chicken & Homestyle Noodles Soup, Organic Light in Sodium Lentil Vegetable Soup, Ultra Soft Facial Tissues, Harvest Cheddar Multigrain Chips, Oven Roasted Turkey Breast"
4,122,Carrots
5,138,"Lemon Cayenne Agave Cold Pressed Juice Beverage, Organic Yellow Onion, Organic Cinnamon Apple Sauce, No Pulp Calcium & Vitamin D Pure Premium 100% Pure Orange Juice, Baby Back Pork Ribs"
6,151,"Peaches, Canola Oil"
7,156,"Classic White Bread, Mild Salsa, Natural Premium Coconut Water, Chocolate Chip Cookie Dough Ice Cream, Organic Free Range Chicken Broth"
8,161,"Oven Roasted Turkey Breast, Vanilla Ice Cream, Chocolate Chip Cookie Dough Ice Cream, Diet Coke Soda, Coffee Ice Cream"
9,165,"Tilapia Filet, Organic Lacinato (Dinosaur) Kale, Natural Free & Clear Dish Liquid, Organic Quinoa Dark Chocolate Bar, Large Lemon, Organic Raw Sharp Cheddar Cheese"


### **Nhận xét**
Ở giai đoạn này, dữ liệu đã được chuyển từ dạng bảng chi tiết sang dạng giao dịch. Cách lưu `product_id` giúp giảm tải tốt hơn, trong khi phần hiển thị vẫn có thể ánh xạ sang `product_name` để diễn giải.


## **39. Tạo ma trận giao dịch ở dạng sparse**

Thay vì dùng `pivot_table` để tạo một ma trận dense rất nặng, notebook chuyển sang dựng ma trận sparse. Đây là lựa chọn phù hợp hơn với dữ liệu market basket vì phần lớn ô trong ma trận sẽ là giá trị 0.


In [38]:
from scipy.sparse import coo_matrix

basket_pairs = du_lieu_loc[["order_id", "product_id"]].drop_duplicates().copy()

ma_order, danh_sach_order = pd.factorize(basket_pairs["order_id"], sort=True)
ma_product, danh_sach_product = pd.factorize(basket_pairs["product_id"], sort=True)

gia_tri = np.ones(len(basket_pairs), dtype="int8")
ma_tran_sparse = coo_matrix(
    (gia_tri, (ma_order, ma_product)),
    shape=(len(danh_sach_order), len(danh_sach_product))
).tocsr()

basket_matrix = pd.DataFrame.sparse.from_spmatrix(
    ma_tran_sparse,
    index=danh_sach_order,
    columns=danh_sach_product
)

basket_matrix.index.name = "order_id"
basket_matrix.columns.name = "product_id"

bang_ma_tran = pd.DataFrame([
    {"Chỉ tiêu": "Số giao dịch (số hàng)", "Giá trị": int(basket_matrix.shape[0])},
    {"Chỉ tiêu": "Số sản phẩm (số cột)", "Giá trị": int(basket_matrix.shape[1])},
    {"Chỉ tiêu": "Mật độ ma trận sparse", "Giá trị": round(float(basket_matrix.sparse.density), 6)}
])

display(dinh_dang_bang(bang_ma_tran, "Bảng 37. Kích thước của basket_matrix dạng sparse"))


,Chỉ tiêu,Giá trị
0,Số giao dịch (số hàng),75519.000000
1,Số sản phẩm (số cột),3000.000000
2,Mật độ ma trận sparse,0.002556


In [39]:
basket_matrix_hien_thi = basket_matrix.head(10).sparse.to_dense().reset_index()

display(dinh_dang_bang(
    basket_matrix_hien_thi,
    "Bảng 38. Minh hoạ một phần của ma trận giao dịch sparse"
))


### **Nhận xét**
Đến đây, dữ liệu đã được chuyển hoàn toàn sang dạng ma trận giao dịch nhưng theo kiểu sparse. Cách biểu diễn này phù hợp hơn nhiều cho bài toán luật kết hợp vì tiết kiệm bộ nhớ hơn so với ma trận dense truyền thống.


## **40. Kiểm tra nhanh ma trận giao dịch**

Sau khi tạo ma trận giao dịch, cần kiểm tra lại một số đặc điểm cơ bản để đảm bảo dữ liệu đầu vào được tạo đúng:
- không có giá trị thiếu,
- giá trị chỉ gồm 0 và 1,
- số giao dịch và số sản phẩm đúng với kỳ vọng sau bước lọc.

In [40]:
kiem_tra_matrix = pd.DataFrame([
    {"Chỉ tiêu": "Số giá trị thiếu trong basket_matrix", "Giá trị": int(basket_matrix.isnull().sum().sum())},
    {"Chỉ tiêu": "Giá trị nhỏ nhất trong ma trận", "Giá trị": int(basket_matrix.min().min())},
    {"Chỉ tiêu": "Giá trị lớn nhất trong ma trận", "Giá trị": int(basket_matrix.max().max())},
    {"Chỉ tiêu": "Số transaction", "Giá trị": int(basket_matrix.shape[0])},
    {"Chỉ tiêu": "Số item", "Giá trị": int(basket_matrix.shape[1])},
    {"Chỉ tiêu": "Mật độ sparse", "Giá trị": round(float(basket_matrix.sparse.density), 6)}
])

display(dinh_dang_bang(kiem_tra_matrix, "Bảng 39. Kiểm tra nhanh chất lượng của basket_matrix"))


,Chỉ tiêu,Giá trị
0,Số giá trị thiếu trong basket_matrix,0.000000
1,Giá trị nhỏ nhất trong ma trận,0.000000
2,Giá trị lớn nhất trong ma trận,1.000000
3,Số transaction,75519.000000
4,Số item,3000.000000
5,Mật độ sparse,0.002556


### **Nhận xét**
Kết quả kiểm tra cho thấy ma trận giao dịch đã được xây dựng đúng dạng nhị phân. Đây là điều kiện cần để có thể sử dụng ma trận này làm đầu vào cho các thuật toán khai thác tập phổ biến và luật kết hợp ở giai đoạn sau.

## **41. Xuất dữ liệu đầu vào cho giai đoạn cuối kì**

Để thuận tiện cho việc sử dụng lại trong giai đoạn cuối kì, dữ liệu trung gian và ma trận giao dịch được lưu lại thành các file riêng.

In [41]:
basket_data_xuat = basket_data.copy()
basket_data_xuat["danh_sach_product_id"] = basket_data_xuat["danh_sach_product_id"].apply(
    lambda x: " | ".join(map(str, x))
)

order_product_pairs = du_lieu_loc[["order_id", "product_id", "product_name"]].drop_duplicates().copy()
product_mapping = du_lieu_loc[["product_id", "product_name", "aisle_name", "department_name"]].drop_duplicates().copy()

basket_data_xuat.to_csv("basket_data_sample.csv", index=False)
order_product_pairs.to_csv("order_product_pairs_sample.csv", index=False)
product_mapping.to_csv("product_mapping_sample.csv", index=False)
basket_matrix.to_pickle("basket_matrix_sparse.pkl")

bang_output_buoc_4 = pd.DataFrame([
    {"Tên file": "basket_data_sample.csv", "Ý nghĩa": "Tập giao dịch theo order_id (danh sách product_id)", "Số dòng": int(basket_data_xuat.shape[0]), "Số cột": int(basket_data_xuat.shape[1])},
    {"Tên file": "order_product_pairs_sample.csv", "Ý nghĩa": "Các cặp order_id - product_id để dựng lại basket", "Số dòng": int(order_product_pairs.shape[0]), "Số cột": int(order_product_pairs.shape[1])},
    {"Tên file": "product_mapping_sample.csv", "Ý nghĩa": "Bảng ánh xạ product_id sang tên/nhóm sản phẩm", "Số dòng": int(product_mapping.shape[0]), "Số cột": int(product_mapping.shape[1])},
    {"Tên file": "basket_matrix_sparse.pkl", "Ý nghĩa": "Ma trận giao dịch sparse dùng cho bước cuối kì", "Số dòng": int(basket_matrix.shape[0]), "Số cột": int(basket_matrix.shape[1])}
])

display(dinh_dang_bang(bang_output_buoc_4, "Bảng 40. Các file đầu ra của bước chuẩn bị dữ liệu đầu vào"))


,Tên file,Ý nghĩa,Số dòng,Số cột
0,basket_data_sample.csv,Tập giao dịch theo order_id (danh sách product_id),75519,2
1,order_product_pairs_sample.csv,Các cặp order_id - product_id để dựng lại basket,579185,3
2,product_mapping_sample.csv,Bảng ánh xạ product_id sang tên/nhóm sản phẩm,3000,4
3,basket_matrix_sparse.pkl,Ma trận giao dịch sparse dùng cho bước cuối kì,75519,3000


## **42. Kết luận Bước 4**

Bước này đã hoàn thành quá trình biến đổi dữ liệu từ bảng tích hợp rút gọn sang dạng giao dịch và ma trận nhị phân phục vụ cho luật kết hợp. So với phiên bản cũ, notebook không còn phụ thuộc vào `instacart_merged_full.csv` hay ma trận dense quá lớn.

Kết quả thu được gồm các tập dữ liệu chính:

- `basket_data_sample.csv`: phản ánh mỗi đơn hàng dưới dạng danh sách `product_id`,
- `order_product_pairs_sample.csv`: tập chi tiết nhẹ để có thể dựng lại giao dịch khi cần,
- `product_mapping_sample.csv`: bảng ánh xạ `product_id` sang tên, quầy hàng và ngành hàng,
- `basket_matrix_sparse.pkl`: ma trận giao dịch sparse dùng trực tiếp cho bước khai thác luật kết hợp.

Như vậy, notebook vẫn giữ nguyên bộ dữ liệu Instacart nhưng đã chuyển sang chiến lược xử lý phù hợp hơn với máy cá nhân.
